# Suite CCHK — Checking C translation units against editions, and as trees

Which editions of C have which constructs, as the port records it: `C.Standards` records the editions `Programs.C`
covers as data, C11, C17 and C23, as the user decided ("Maybe C11, C17, C23 Cpp17, cpp20, cpp23", C17 being C11's
corrigenda, with no new syntax), each holding the statement, declaration and member kinds it has, as
mbse-expressions' C dialects record the terms' (mbse-basis CAP-268); `C.Check.check(store, path, version)` lists the
constructs a stored unit uses that an edition lacks, each located by where it stands in the unit's tree (`body[0]`,
`body[1].declarators[0].initializer`), reading those standards and the grammar clauses they allow or forbid, the
language's own, its statements' (`Standards.GRAMMAR_CLAUSES`), and its dialects', as data;
`C.Check.validate(store, path)` lists a unit's problems as a tree, each kind's own rules and its terms' as
their dialects validate them. GNU's extensions are in no edition, and not checked. Where a statement or a member
declaration may stand is no rule of `validate`'s: the schemas refuse a misplaced one on write. Ported from 0.3's C
cases of CPP (where kinds and features exist, CPP-03 and CPP-04, within C) and PRT (printing checked the standard).

A unit's tree is one value, its `body`, each statement a value object; where a term holds statements, a statement
expression's `block`, a struct's `members` and a conditional directive's `group`, `check` and `validate` read them as
statements, located through the term's place. Each case parses source with `C.Parse.parse` into one store,
`C.store()` prepared by `Standards.prepare`, or, where the parser would not write the tree, writes the unit's `body` by
hand in mbse-schemas' builder syntax, as `Parse` writes it: a list of branches of `Programs.C.Statement`, each tagged
by its kind in lower case (`ifstatement`), the terms nested in it as values of mbse-expressions' unions (`a` below
writes the variable `a`).

In [ ]:
import itertools
from pathlib import Path

from mbse.Expressions.C.Expressions import Expressions as CExpressions
from mbse.Expressions.Framework import Objects, Standards as ExpressionStandards, Terms
from mbse.Programs import C
from mbse.Programs.C import Check, Parse, Print, Standards
from mbse.Programs.C.Statements import KINDS, LANGUAGE
from mbse.Programs.Framework import Model
from support import raises

SOURCES = Path("../../conformance/sources")
STORE = Standards.prepare(C.store())
PATHS = (f"case{n}.c" for n in itertools.count())


def parsed(text):
    """The path of a new unit holding `text`, parsed into the store."""
    path = next(PATHS)
    STORE.transaction(Parse.parse(text, path))
    return path


def check(text, version):
    return Check.check(STORE, parsed(text), version)


def needs(text, *problems):
    """`text` is fine in C23, and has `problems` in C17 and C11."""
    path = parsed(text)
    assert Check.check(STORE, path, "C23") == [], Check.check(STORE, path, "C23")
    assert Check.check(STORE, path, "C17") == list(problems), Check.check(STORE, path, "C17")
    assert Check.check(STORE, path, "C11") == list(problems), Check.check(STORE, path, "C11")


def written(*body):
    """The path of a new unit whose `body` is the statements `body`, each written in builder syntax as a branch of
    `Programs.C.Statement`, as `Parse.parse` writes one."""
    path = next(PATHS)

    def transaction(txn):
        return txn.uid(path).mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
            .symbols({"ws": txn.Programs.Framework.Workspace, "unit": txn.Programs.C.TranslationUnit})
            .spec([
                lambda _: _.ws.files({path: lambda e: e.file(_.unit)}),
                lambda _: _.unit.body(list(body)),
            ])]))])

    STORE.transaction(transaction)
    return path


def validate(*body):
    return Check.validate(STORE, written(*body))


# Terms and statements the cases share, in the builders' syntax: three variables, a name and `int`; `;` and `b;`
a = lambda t: t.variable(lambda v: v.name("a"))  # noqa: E731
b = lambda t: t.variable(lambda v: v.name("b"))  # noqa: E731
c = lambda t: t.variable(lambda v: v.name("c"))  # noqa: E731
NAME = lambda n: n.identifier(lambda i: i.spelling("x"))  # noqa: E731
INT = lambda p: p.primitivetypespecifier(lambda k: k.keyword("int"))  # noqa: E731
EMPTY = lambda u: u.expressionstatement(lambda s: s)  # noqa: E731
SAY_B = lambda u: u.expressionstatement(lambda s: s.expression(b))  # noqa: E731

## CCHK-01 · The editions as data: `Programs.C` covers C11, C17 and C23, each holding its statement, declaration and member kinds, by membership, beside mbse-expressions' dialects' own, and its statements' grammar clauses, C23's, the K&R definition it removed, and those nested or required, an edition it does not represent guessed by its neighbours' flags, one no edition names allowed; run again, nothing changes

In [ ]:
DESIGNATIONS = ["ISO/IEC 9899:2011", "ISO/IEC 9899:2018", "ISO/IEC 9899:2024"]
with STORE.transaction():
    language = STORE.singleton(LANGUAGE)
    standards = ExpressionStandards.standards(language)
    assert list(standards) == DESIGNATIONS
    assert ExpressionStandards.find(language, "C23") is standards["ISO/IEC 9899:2024"]
    assert ExpressionStandards.properties(standards["ISO/IEC 9899:2018"]) == {
        "bodies": ("org.iso", "ch.iec"), "number": "9899", "revision": ("year", 2018), "names": ("C17",)}
    assert ExpressionStandards.designation_of(standards["ISO/IEC 9899:2011"]) == "ISO/IEC 9899:2011"

    def kinds(designation):
        return {kind.name.rsplit(".", 1)[-1] for kind in ExpressionStandards.kinds(standards[designation])}

    assert len(KINDS) == 23 and kinds(DESIGNATIONS[2]) == set(KINDS) - {"AsmDeclaration"}  # GNU's `asm`, in none
    assert kinds(DESIGNATIONS[0]) == kinds(DESIGNATIONS[1]) == kinds(DESIGNATIONS[2]) - {"AttributeDeclaration"}
    assert "Error" not in kinds(DESIGNATIONS[2])  # invalid syntax, in none
    # The terms' editions are mbse-expressions' own, each dialect's by the same designations
    for dialect in Standards.DIALECTS[1:]:
        assert list(ExpressionStandards.standards(STORE.singleton(dialect)))[:3] == list(standards), dialect
    expressions = ExpressionStandards.standards(STORE.singleton("Expressions.C.Expressions"))
    tags = {kind.name for kind in ExpressionStandards.kinds(expressions["ISO/IEC 9899:2011"])}
    assert "Expressions.C.Expressions.GenericSelection" in tags
    assert "Expressions.C.Expressions.StatementExpression" not in tags  # GNU's, in none
# The language is one of the framework's dialects, which is what may own a standard
dialects = STORE.registered("Expressions.Framework.Dialect")
assert any(b.name == "programs_c" and b.schema.name == "Programs.C.Language" for b in dialects.branches)
# Run again, each writes nothing: the same editions, each with the same kinds
Standards.prepare(STORE)
with STORE.transaction():
    again = ExpressionStandards.standards(STORE.singleton(LANGUAGE))
    assert list(again) == list(standards) and len(ExpressionStandards.kinds(again["ISO/IEC 9899:2024"])) == 22
assert Standards.ONLY == {"AttributeDeclaration": ("C23",)} and Standards.EXTENSIONS == ("AsmDeclaration",)
assert Standards.EDITIONS["C17"] == ("ISO/IEC 9899:2018", 2018) and not hasattr(Standards, "SINCE")  # membership
# What `define` writes from `ONLY`, `check` reads: a kind recorded in later editions alone is lacking before
Standards.ONLY["StaticAssertDeclaration"] = ("C17", "C23")
try:
    later = Standards.prepare(C.store())
finally:
    del Standards.ONLY["StaticAssertDeclaration"]
later.transaction(Parse.parse("_Static_assert(1, \"m\");\n", "later.c"))
assert Check.check(later, "later.c", "C17") == []
assert Check.check(later, "later.c", "C11") == ["body[0]: StaticAssertDeclaration needs C17"]
# The language's grammar clauses, its statements', owned by name through its adjacency `grammar_clauses` (added by
# schema extension, as `standards` is): those an edition names, each allowed by C23 and forbidden by C11 and C17, the
# oldest's entry flagged `before` and the newest's `after`, but the K&R definition, which C23 removed; and those nested
# or required, which no edition names, so each allows; named as `check` names the language feature
assert {"standards", "grammar_clauses"} <= set(STORE.registered(f"{LANGUAGE}.Language").adjacencies)
LABELS = ("LabeledStatement", "CaseStatement", "DefaultStatement")
SPECIFIED = ("SimpleDeclaration", "MemberDeclaration", "FunctionDefinition")
NAMES = [*(f"{label} without a statement" for label in LABELS), *(f"{label} before a declaration" for label in LABELS),
         "StaticAssertDeclaration.keyword 'static_assert'", "StaticAssertDeclaration without a message",
         *(f"{holder}.specifiers holding a {helper}" for holder in SPECIFIED
           for helper in ("bitintspecifier", "placeholdertypespecifier", "standardattributespecifier")),
         *(f"{holder}.attributes holding a standardattributespecifier"
           for holder in (*SPECIFIED, "AttributedStatement")),
         "SimpleDeclaration with an inferred type", "FunctionDefinition with an unnamed parameter"]
REMOVED = "FunctionDefinition with a declaration list"
with STORE.transaction():
    language = STORE.singleton(LANGUAGE)
    clauses = ExpressionStandards.grammar_clauses(language)
    assert list(clauses) == [clause.name for clause in Standards.GRAMMAR_CLAUSES] and len(clauses) == 38
    editions = ExpressionStandards.standards(language)
    allowing = {name: [ExpressionStandards.tags(s)[-1] for s in editions.values() if ExpressionStandards.allows(
        s, clause)] for name, clause in clauses.items()}
    named = [name for name, clause in clauses.items() if Objects.entries(clause, "allowed")]
    assert named == [*NAMES, REMOVED] and len(NAMES) == 23
    for name in NAMES:
        assert allowing[name] == ["C23"], name
        assert [e.values for e in Objects.entries(clauses[name], "forbidden")] == [{"before": True}, {}], name
        assert [e.values for e in Objects.entries(clauses[name], "allowed")] == [{"after": True}], name
    assert allowing[REMOVED] == ["C11", "C17"]  # removed: C11's entry, allowing, flagged `before`, C23's `after`
    assert [e.values for e in Objects.entries(clauses[REMOVED], "allowed")] == [{"before": True}, {}]
    assert [e.values for e in Objects.entries(clauses[REMOVED], "forbidden")] == [{"after": True}]
    for name in set(clauses) - set(named):  # nested or required: no edition names them, so each allows them
        assert allowing[name] == ["C11", "C17", "C23"] and not Objects.entries(clauses[name], "forbidden"), name
    KIND = lambda name: STORE.registered(f"{LANGUAGE}.{name}")  # noqa: E731
    assert ExpressionStandards.grammar_clause(clauses["LabeledStatement without a statement"]) == {
        "kind": KIND("LabeledStatement"), "property": "statement", "present": False}
    assert ExpressionStandards.grammar_clause(clauses["CaseStatement before a declaration"]) == {
        "kind": KIND("CaseStatement"), "property": "statement",
        "kinds": tuple(KIND(name) for name in ("SimpleDeclaration", "StaticAssertDeclaration", "AttributeDeclaration"))}
    assert ExpressionStandards.grammar_clause(clauses["StaticAssertDeclaration.keyword 'static_assert'"]) == {
        "kind": KIND("StaticAssertDeclaration"), "property": "keyword", "values": ("static_assert",)}
    assert ExpressionStandards.grammar_clause(clauses["MemberDeclaration.specifiers holding a bitintspecifier"])[
        "kinds"] == (STORE.registered("Expressions.C.Types.BitIntSpecifier"),)  # C Types' helper
# An edition the data does not represent, C2y, is guessed to allow each, as C23's entry is flagged `after`; one before
# C11, to forbid each, as C11's is flagged `before`: each written in a store of its own, as the language would cover it
with later.transaction():
    Standard = later.Expressions.Framework.Standard
    c2y = Standard(lambda s: s.bodies(["org.iso", "ch.iec"]).number("9899").revision(lambda r: r.year(2029)).names(
        ["C2y"]))
    c99 = Standard(lambda s: s.bodies(["org.iso", "ch.iec"]).number("9899").revision(lambda r: r.year(1999)).names(
        ["C99"]))
    later.Programs.C.Language(later.singleton(LANGUAGE), lambda b: b.standards({
        "ISO/IEC 9899:2029": lambda e: e.standard(c2y), "ISO/IEC 9899:1999": lambda e: e.standard(c99)}))
    owned = ExpressionStandards.grammar_clauses(later.singleton(LANGUAGE))
    assert all(ExpressionStandards.allows(c2y, owned[name]) for name in NAMES)
    assert not any(ExpressionStandards.allows(c99, owned[name]) for name in NAMES)
    assert ExpressionStandards.allows(c99, owned[REMOVED]) and not ExpressionStandards.allows(c2y, owned[REMOVED])
# A grammar clause no edition names is allowed by each: one the language owns, written beside its own
with later.transaction():
    plain = later.Expressions.Framework.GrammarClause(lambda c: c.property("keyword").values(["_Static_assert"]).kind(
        [later.registered(f"{LANGUAGE}.StaticAssertDeclaration")]))
    later.Programs.C.Language(later.singleton(LANGUAGE), lambda b: b.grammar_clauses({
        "StaticAssertDeclaration.keyword '_Static_assert'": lambda e: e.clause(plain)}))
    assert all(ExpressionStandards.allows(s, plain) for s in ExpressionStandards.standards(
        later.singleton(LANGUAGE)).values())
assert Check.check(later, "later.c", "C17") == []  # taken, and allowed
print("ok")

## CCHK-02 · Where language features exist: C23's, each by data, the language's grammar clauses and its dialects', each located by where it stands in the unit's tree, each once; what C11 has, every edition has

In [ ]:
# The language's grammar clauses, its statements'
needs("static_assert(1, \"m\");\n_Static_assert(1);\n",
      "body[0]: StaticAssertDeclaration.keyword 'static_assert' needs C23",
      "body[1]: StaticAssertDeclaration without a message needs C23")
needs("void f(int n) {\n    switch (n) {\n    case 1:\n    }\n    switch (n) {\n    default:\n    }\nend:\n}\n",
      "body[0].body.body[0].body.body[0]: CaseStatement without a statement needs C23",
      "body[0].body.body[1].body.body[0]: DefaultStatement without a statement needs C23",
      "body[0].body.body[2]: LabeledStatement without a statement needs C23")
needs("void f(void) {\nfirst:\n    int x = 0;\n}\n",
      "body[0].body.body[0]: LabeledStatement before a declaration needs C23")
needs("struct s { _BitInt(4) b; [[deprecated]] int d; };\n[[noreturn]] void f(void) { [[maybe_unused]] l: ; }\n",
      "body[0].specifiers[0].members.members[0]: MemberDeclaration.specifiers holding a bitintspecifier needs C23",
      "body[0].specifiers[0].members.members[1]: MemberDeclaration.attributes holding a standardattributespecifier needs "
      "C23", "body[1]: FunctionDefinition.attributes holding a standardattributespecifier needs C23",
      "body[1].body.body[0]: AttributedStatement.attributes holding a standardattributespecifier needs C23")
# What no grammar clause could express, once code, is data: an unnamed parameter of a definition's own (CCHK-11)
assert not hasattr(Check, "FEATURES") and not hasattr(Check, "STATEMENT_FEATURES")  # the code tables, gone
assert not hasattr(Check, "UNRECORDED")
needs("int f(int a, int) { return a; }\nint (*g(int, int b))(void) { return 0; }\n",
      "body[0]: FunctionDefinition with an unnamed parameter needs C23",
      "body[1]: FunctionDefinition with an unnamed parameter needs C23")  # its own, not its type's
assert check("int f(void) { return 0; }\nint g(int a, ...) { return a; }\nint h(int);\n"
             "void (*k(int a))(int) { return 0; }\nint m { return 0; }\n", "C11") == []
# The dialects' grammar clauses: C's, C Types' and C Preprocessor's, for the terms and helpers a statement holds
needs("thread_local int a;\nconstexpr bool b = true;\n_Decimal32 d;\n_Float16 h;\n",
      "body[0].specifiers[0]: declspecifier.keyword 'thread_local' needs C23",
      "body[1].specifiers[0]: declspecifier.keyword 'constexpr' needs C23",
      "body[1].specifiers[1]: primitivetypespecifier.keyword 'bool' needs C23",
      "body[1].declarators[0].initializer: literal true or false needs C23",
      "body[2].specifiers[0]: primitivetypespecifier.keyword '_Decimal32' needs C23",
      "body[3].specifiers[0]: primitivetypespecifier.keyword '_Float16' needs C23")
needs("typeof(1) a;\n_BitInt(8) b = 0wb;\nauto c = 0b101;\nenum e : int { E };\n",
      "body[0].specifiers[0]: typeofspecifier.keyword 'typeof' needs C23",
      "body[1]: SimpleDeclaration.specifiers holding a bitintspecifier needs C23",
      "body[1].declarators[0].initializer: literal.suffix 'wb' needs C23",
      "body[2]: SimpleDeclaration with an inferred type needs C23",
      "body[2].declarators[0].initializer: literal.base 2 needs C23",
      "body[3].specifiers[0]: enumspecifier.base needs C23")
# `auto` as a type, which `Parse` writes as the keyword `auto`, written as the helper C Types has for it
path = written(lambda u: u.simpledeclaration(lambda s: s.specifiers([lambda p: p.placeholdertypespecifier(
    lambda k: k)]).declarators([lambda d: d.declarator(lambda k: k.iddeclarator(lambda i: i.name(NAME)))])))
assert Check.check(STORE, path, "C23") == []
assert Check.check(STORE, path, "C17") == [
    "body[0]: SimpleDeclaration.specifiers holding a placeholdertypespecifier needs C23"]
needs("[[nodiscard]] int f(void);\nalignas(8) char c = u8'c';\nint s = alignof(int) + 1'000;\nint z[2] = {};\n",
      "body[0]: SimpleDeclaration.attributes holding a standardattributespecifier needs C23",
      "body[1].specifiers[0]: alignasspecifier.keyword 'alignas' needs C23",
      "body[1].declarators[0].initializer: literal.prefix 'u8' of a character needs C23",  # mbse-expressions' data
      "body[2].declarators[0].initializer: alignofexpression.keyword 'alignof' needs C23",
      "body[2].declarators[0].initializer: literal with digit separators needs C23",
      "body[3].declarators[0].initializer: initializerlist without items needs C23")
needs("#warning \"w\"\n#embed \"e\"\n#ifdef A\n#elifdef B\n#endif\nvoid *p = nullptr;\n_Decimal64 m = 1.5dd;\n",
      "body[0].directive: otherdirective.directive 'warning' needs C23",
      "body[1].directive: otherdirective.directive 'embed' needs C23",
      "body[2].directive: elifdefdirective needs C23",  # the dialects' data: `#elifdef`
      "body[3].declarators[0].initializer: constant needs C23",  # and `nullptr`
      "body[4].specifiers[0]: primitivetypespecifier.keyword '_Decimal64' needs C23",
      "body[4].declarators[0].initializer: literal with a decimal floating suffix needs C23")
needs("#if __has_include(<x.h>) || defined(X)\n#elif __has_include(M)\n#endif\n",  # the dialects' data: once
      "body[0].directive: hasincludeexpression needs C23")  # `__has_include`, C23's, not C++17's, in C
needs("int f(void) { [[fallthrough]]; return (int)sizeof(int[3]){1, 2, 3}; }\n",
      "body[0].body.body[0]: AttributeDeclaration needs C23")  # the language's data: its kind, C23's alone
needs("int a = f(alignof(int), alignof(long));\n",  # one term: once
      "body[0].declarators[0].initializer: alignofexpression.keyword 'alignof' needs C23")
# What C11 has, every edition has: `_Alignas`, `_Alignof`, `_Atomic`, `_Generic`, `_Noreturn`, `_Static_assert`,
# `_Thread_local`, `restrict`, `inline`, compound literals, designators, `u8` strings, and GNU's extensions, in none
assert check("_Alignas(8) char c;\n_Thread_local int t;\n_Atomic int at;\n_Noreturn void stop(void);\n"
             "inline int f(int *restrict p) { return _Generic(*p, int: 1, default: 0) + _Alignof(int); }\n"
             "_Static_assert(1, \"m\");\nchar *s = u8\"text\";\nstruct p { int x; } o = (struct p){.x = 1};\n"
             "int g(void) { return ({ 1; }) + __extension__ 2; }\nint h(int n) { switch (n) case 1 ... 2: ; "
             "asm(\"nop\"); return 0; }\n__typeof__(1) gnu;\nint x __attribute__((aligned(4)));\n", "C11") == []
# GNU's labels as values and asm labels, in none; `_Pragma` in every edition, C Preprocessor's data
assert check('int x asm("r1");\nint f(void) __asm__("g");\n_Pragma("once")\n'
             'void *h(void) {\n    _Pragma("GCC unroll 2")\n    _Pragma("x");\nl:\n    return &&l;\n}\n', "C11") == []
with STORE.transaction():
    for designation in ("ISO/IEC 9899:2011", "ISO/IEC 9899:2024"):
        held = {kind.name for dialect in Standards.DIALECTS for kind in ExpressionStandards.kinds(
            ExpressionStandards.standards(STORE.singleton(dialect))[designation])}
        assert "Expressions.C.Preprocessor.PragmaExpression" in held
        assert {"Expressions.C.Expressions.LabelAddressExpression", "Expressions.C.Types.AsmLabel"}.isdisjoint(held)
print("ok")

## CCHK-03 · The edition by its number, its common name or its designation; editions the language does not cover, a store not prepared, and a path the workspace does not hold

In [ ]:
path = parsed("static_assert(1);\n")
problems = ["body[0]: StaticAssertDeclaration.keyword 'static_assert' needs C23",
            "body[0]: StaticAssertDeclaration without a message needs C23"]
for version in ((11,), "C11", "ISO/IEC 9899:2011", (17,), "C17", "ISO/IEC 9899:2018"):
    assert Check.check(STORE, path, version) == problems, version
for version in ((23,), "C23", "ISO/IEC 9899:2024"):
    assert Check.check(STORE, path, version) == [], version
for version in ((99,), (2023,), (1, 1), "C99", "C++23", "ISO/IEC 14882:2024", "2024"):
    with raises(LookupError, match="Programs.C covers no edition"):
        Check.check(STORE, path, version)
with raises(LookupError):
    Check.check(STORE, "nowhere.c", "C23")
with raises(LookupError):
    Check.validate(STORE, "nowhere.c")
# Within the caller's transaction, a unit's visitor is read; the store and a path open their own
with STORE.transaction():
    unit = STORE.singleton(Model.WORKSPACE).files({"path": path})
    assert Check.check_file(STORE, unit, "C11") == problems
    assert Check.validate_file(unit) == []
    with raises(ValueError, match="transactions do not nest"):
        Check.check(STORE, path, "C11")
# A store that holds the installs alone has no editions; one with the language's alone has no terms' editions
bare = C.store()
bare.transaction(Parse.parse("static_assert(1);\n", "a.c"))
with raises(LookupError, match="run Standards.prepare"):
    Check.check(bare, "a.c", "C23")
bare.transaction(Standards.install)
bare.transaction(Standards.define)
with raises(LookupError, match="Expressions.C.Expressions covers no edition"):
    Check.check(bare, "a.c", "C23")
assert Check.check(Standards.prepare(bare), "a.c", "C23") == []
print("ok")

## CCHK-04 · The conformance source: none of its constructs lacking in C23 but its K&R definition, which C23 removed; in C17 and C11, each of C23's where it stands

In [ ]:
text = (SOURCES / "c23.c").read_text(encoding="utf-8")
path = parsed(text)
assert Check.check(STORE, path, "C23") == Check.check(STORE, path, (23,)) == [
    "body[54]: FunctionDefinition with a declaration list needs C17"]  # its K&R definition, which C23 removed
oldest = Check.check(STORE, path, "C11")
assert Check.check(STORE, path, "C17") == oldest
assert oldest[:4] == ["body[9].directive: otherdirective.directive 'embed' needs C23", "body[11].directive: hasincludeexpression needs C23",
                      "body[12].directive: elifdefdirective needs C23",
                      "body[17].specifiers[0]: declspecifier.keyword 'thread_local' needs C23"], oldest
assert "body[63].specifiers[0].members.members[5]: StaticAssertDeclaration without a message needs C23" in oldest
assert "body[70].body.body[7].body.body[3]: CaseStatement before a declaration needs C23" in oldest  # `case 5:`
assert "body[29]: SimpleDeclaration.specifiers holding a bitintspecifier needs C23" in oldest  # the language's data
assert len(oldest) == 40 and all(problem.endswith("needs C23") for problem in oldest), oldest
assert Check.validate(STORE, path) == []
print(len(oldest), "constructs lacking in C11")

## CCHK-05 · Each kind's own rules: what it needs, what its lists must hold, what goes together, the words its properties may be, and the dangling `else`; where a statement or a member declaration stands, the schemas enforce on write

A block (`body`) is a `Programs.C.Body`, whose union has no member declaration; a struct's `members` take only member
declarations, static assertions, directives, comments and errors; a kind has only its own properties. So a misplaced
statement or member declaration is never stored: writing one fails, an `AttributeError` for a branch the union does
not have, and `validate` has no rule for it.

In [ ]:
DECLARE_X = lambda d: d.declarator(lambda k: k.iddeclarator(lambda i: i.name(NAME)))  # noqa: E731
FUNCTION_X = lambda k: k.functiondeclarator(lambda f: f.declarator(lambda k: k.iddeclarator(  # noqa: E731
    lambda i: i.name(NAME))))
problems = [
    (lambda u: u.comment(lambda s: s), "body[0]: a Comment needs a text"),
    (lambda u: u.directive(lambda s: s), "body[0]: a Directive needs a directive"),
    (lambda u: u.ifstatement(lambda s: s.consequence(EMPTY)), "body[0]: an IfStatement needs a condition"),
    (lambda u: u.ifstatement(lambda s: s.condition(a)), "body[0]: an IfStatement needs a consequence"),
    (lambda u: u.switchstatement(lambda s: s.body(EMPTY)), "body[0]: a SwitchStatement needs a condition"),
    (lambda u: u.whilestatement(lambda s: s.condition(a)), "body[0]: a WhileStatement needs a body"),
    (lambda u: u.dostatement(lambda s: s.body(EMPTY)), "body[0]: a DoStatement needs a condition"),
    (lambda u: u.forstatement(lambda s: s.condition(a)), "body[0]: a ForStatement needs a body"),
    (lambda u: u.labeledstatement(lambda s: s.statement(EMPTY)), "body[0]: a LabeledStatement needs a label"),
    (lambda u: u.casestatement(lambda s: s.last(a)), "body[0]: a CaseStatement needs a value"),
    (lambda u: u.gotostatement(lambda s: s), "body[0]: a GotoStatement needs a label or a target"),
    (lambda u: u.attributedstatement(lambda s: s.statement(EMPTY)),
     "body[0]: an AttributedStatement needs an attribute"),
    (lambda u: u.asmdeclaration(lambda s: s.keyword("asm")), "body[0]: an AsmDeclaration needs a template"),
    (lambda u: u.asmdeclaration(lambda s: s.keyword("assembly").template(a)),
     "body[0]: AsmDeclaration.keyword cannot be 'assembly'"),
    (lambda u: u.asmdeclaration(lambda s: s.keyword("asm").template(a).outputs([lambda o: o.value(b)])),
     "body[0].outputs[0]: an AsmOperand needs a constraint"),
    (lambda u: u.simpledeclaration(lambda s: s.declarators([DECLARE_X])),
     "body[0]: a SimpleDeclaration needs a specifier"),
    (lambda u: u.simpledeclaration(lambda s: s.specifiers([INT]).declarators([lambda d: d.initializer(a)])),
     "body[0].declarators[0]: an InitDeclarator needs a declarator"),
    (lambda u: u.functiondefinition(lambda f: f.specifiers([INT]).declarator(FUNCTION_X)),
     "body[0]: a FunctionDefinition needs a body"),
    (lambda u: u.functiondefinition(lambda f: f.declarator(FUNCTION_X).body(lambda k: k)),
     "body[0]: a FunctionDefinition needs a specifier"),  # C99 removed the implicit `int`
    (lambda u: u.staticassertdeclaration(lambda s: s.keyword("static_assert")),
     "body[0]: a StaticAssertDeclaration needs a condition"),
    (lambda u: u.staticassertdeclaration(lambda s: s.keyword("assert").condition(a)),
     "body[0]: StaticAssertDeclaration.keyword cannot be 'assert'"),
    (lambda u: u.attributedeclaration(lambda s: s), "body[0]: an AttributeDeclaration needs an attribute"),
    (lambda u: u.simpledeclaration(lambda s: s.specifiers([lambda p: p.classspecifier(lambda k: k.key("struct").body(
        True).members(lambda m: m.memberlist(lambda m: m.members([lambda m: m.memberdeclaration(
            lambda d: d.declarators([DECLARE_X]))]))))])),
     "body[0].specifiers[0].members.members[0]: a MemberDeclaration needs a specifier"),
    (lambda u: u.simpledeclaration(lambda s: s.specifiers([lambda p: p.classspecifier(lambda k: k.key("struct").body(
        True).members(lambda m: m.memberlist(lambda m: m.members([lambda m: m.memberdeclaration(lambda d: d.specifiers(
            [INT]).declarators([lambda d: d]))]))))])),
     "body[0].specifiers[0].members.members[0].declarators[0]: a MemberDeclarator needs a declarator or a bitfield"),
]
for body, problem in problems:
    assert validate(body) == [problem], (problem, validate(body))
# `if (a) if (b) ; else ;` would give the else to the inner `if`, through loops, labels and attributes too
dangling = "body[0]: an if without else ends the consequence, and would take the else"
for consequence in (lambda k: k.ifstatement(lambda s: s.condition(b).consequence(EMPTY)),
                    lambda k: k.labeledstatement(lambda s: s.label(lambda i: i.spelling("l")).statement(
                        lambda k: k.forstatement(lambda f: f.body(lambda k: k.ifstatement(
                            lambda s: s.condition(b).consequence(EMPTY)))))),
                    lambda k: k.ifstatement(lambda s: s.condition(b).consequence(EMPTY).alternative(
                        lambda k: k.whilestatement(lambda w: w.condition(c).body(lambda k: k.ifstatement(
                            lambda s: s.condition(c).consequence(EMPTY))))))):
    assert validate(lambda u: u.ifstatement(lambda s: s.condition(a).consequence(consequence).alternative(
        EMPTY))) == [dangling]
for consequence in (lambda k: k.dostatement(lambda s: s.condition(b).body(EMPTY)),
                    lambda k: k.ifstatement(lambda s: s.condition(b).consequence(EMPTY).alternative(EMPTY))):
    assert validate(lambda u: u.ifstatement(lambda s: s.condition(a).consequence(consequence).alternative(EMPTY))) == []
assert validate(lambda u: u.ifstatement(lambda s: s.condition(a).consequence(lambda k: k.ifstatement(
    lambda s: s.condition(b).consequence(EMPTY))))) == []  # no else to take
assert validate(lambda u: u.ifstatement(lambda s: s.condition(a).consequence(lambda k: k.whilestatement(
    lambda s: s.condition(b))).alternative(EMPTY))) == ["body[0].consequence: a WhileStatement needs a body"]
assert validate(lambda u: u.gotostatement(lambda s: s.target(a)), EMPTY, SAY_B) == []  # GNU's `goto *a;`
# A function's definition whose declarator declares no function, which the schemas hold, and which `Parse` keeps as
# invalid syntax where tree-sitter reads one (its departure 36)
assert validate(lambda u: u.functiondefinition(lambda f: f.specifiers([INT]).declarator(lambda k: k.iddeclarator(
    lambda i: i.name(NAME))).body(lambda k: k))) == ["body[0]: a FunctionDefinition's declarator declares no function"]
assert Check.validate(STORE, parsed("int m { return 0; }\nint (n)(void) { return 0; }\nint f() { return 0; }\n")) == [
    "body[0]: invalid syntax 'int m { return 0; }'"]
assert validate(lambda u: u.simpledeclaration(lambda s: s.specifiers([lambda p: p.classspecifier(lambda k: k.key(
    "struct").body(True).members(lambda m: m.memberlist(lambda m: m.members([lambda m: m.memberdeclaration(
        lambda d: d.specifiers([INT]).declarators([lambda d: d.bitfield(lambda t: t.literal(
            lambda l: l.value(0).text("0").base(10)))]))]))))]))) == []  # `int : 0;`
# Where a statement or a member declaration stands, the schemas enforce: each out of place
with raises(AttributeError, match="memberdeclaration"):
    written(lambda u: u.memberdeclaration(lambda d: d.specifiers([INT])))
with raises(AttributeError, match="expressionstatement"):
    written(lambda u: u.simpledeclaration(lambda s: s.specifiers([lambda p: p.classspecifier(lambda k: k.key(
        "struct").members(lambda m: m.memberlist(lambda m: m.members([SAY_B]))))])))
print(len(problems), "problems")

## CCHK-06 · Terms as their dialects validate them, located at the outermost term, helper or part that holds them; a helper a statement holds alone, by its own checks; names they refer to taken as bound

In [ ]:
BAD = lambda t: t.memberexpression(lambda m: m.object(a).operator("::").member(  # noqa: E731
    lambda n: n.identifier(lambda i: i.spelling("m"))))
assert validate(lambda u: u.expressionstatement(lambda s: s.expression(BAD))) == [
    "body[0].expression: a memberexpression's operator must be '.' or '->', got '::'"]
assert validate(lambda u: u.returnstatement(lambda s: s.value(lambda t: t.variable(lambda v: v.name("1x"))))) == [
    "body[0].value: '1x' is not an identifier"]
assert validate(lambda u: u.simpledeclaration(lambda s: s.specifiers([INT]).declarators([lambda d: d.declarator(
    lambda k: k.iddeclarator(lambda i: i.name(NAME))).initializer(BAD)]))) == [
    "body[0].declarators[0].initializer: a memberexpression's operator must be '.' or '->', got '::'"]  # in a part
# Helpers a statement holds alone: a specifier, a declarator, an attribute specifier, each by its own checks and its
# terms'
assert validate(lambda u: u.simpledeclaration(lambda s: s.specifiers([lambda p: p.namedtypespecifier(
    lambda n: n.name(lambda n: n.identifier(lambda i: i.spelling("2t"))))]).declarators([lambda d: d.declarator(
        lambda k: k.arraydeclarator(lambda r: r.declarator(lambda k: k.iddeclarator(lambda i: i.name(NAME))).size(
            BAD)))]))) == [
    "body[0].specifiers[0]: '2t' is not an identifier",
    "body[0].declarators[0].declarator: a memberexpression's operator must be '.' or '->', got '::'"]
assert validate(lambda u: u.attributedeclaration(lambda s: s.attributes([lambda a: a.standardattributespecifier(
    lambda z: z.attributes([lambda t: t.name("deprecated").arguments([BAD])]))]))) == [
    "body[0].attributes[0]: a memberexpression's operator must be '.' or '->', got '::'"]
assert validate(lambda u: u.directive(lambda s: s.directive(lambda d: d.includedirective(lambda i: i.directive(
    "include"))))) == ["body[0].directive: an IncludeDirective has a path or a macro"]  # C Preprocessor's own
# Parsed units: names are the unit's to bind, so a term's names are taken as bound; a type C leaves to the
# implementation is not checked against its range, which only an ABI gives
assert Check.validate(STORE, parsed("int f(int a) { return a + b + g(c); }\nchar c = 300;\nint i = 1u;\n")) == []
assert Check.validate(STORE, parsed("_Decimal32 d = 1e999df;\n")) == []
print("ok")

## CCHK-07 · Printing writes what the store holds; checking it against an edition is `check`'s, and validating it `validate`'s (ported from PRT)

In [ ]:
path = parsed("static_assert(1);\n")
assert Print.print_module(STORE, path) == "static_assert(1);\n"  # printed, whatever the edition
assert Check.check(STORE, path, "C17") == ["body[0]: StaticAssertDeclaration.keyword 'static_assert' needs C23",
                                           "body[0]: StaticAssertDeclaration without a message needs C23"]
path = written(lambda u: u.ifstatement(lambda s: s.condition(a).consequence(lambda k: k.ifstatement(
    lambda s: s.condition(b).consequence(SAY_B))).alternative(SAY_B)))
assert Print.print_module(STORE, path) == "if (a)\n    if (b)\n        b;\nelse\n    b;\n"  # printed as held
assert Check.validate(STORE, path) == ["body[0]: an if without else ends the consequence, and would take the else"]
print("ok")

## CCHK-08 · Invalid syntax, which a parse keeps: `validate` reports each `Error` and each `error` and `missing` term, located; `check` skips them and checks what was recovered within an `Error`; an `Error` stands where a statement or a member declaration may, and holds statements

In [ ]:
path = parsed("int a = 1;\n"
              "int f(void) {\n"
              "    int x = = 1;\n"
              "    int y = (a + );\n"
              "}\n"
              "struct s { int = ; int ok; };\n"
              "x = 1;\n")
assert Check.validate(STORE, path) == [
    "body[1].body.body[0]: invalid syntax 'int x = = 1;'",
    "body[1].body.body[1].declarators[0].initializer: expression: right: missing identifier",
    "body[2].specifiers[0].members.members[0]: invalid syntax 'int = ;'",  # where a member declaration stands
    "body[3]: invalid syntax 'x = 1;'",  # a statement at file scope, which C does not have
], Check.validate(STORE, path)
assert Check.check(STORE, path, "C11") == []  # skipped: no edition lists invalid syntax
recovered = parsed("void f(void) {\n    if (a {\n        static_assert(1);\n    }\n}\n")
assert Check.validate(STORE, recovered) == [
    "body[0].body.body[0]: invalid syntax 'if (a {\\n        static_assert(1);\\n    }'"]
assert Check.check(STORE, recovered, "C17") == [
    "body[0].body.body[0].body[0]: StaticAssertDeclaration.keyword 'static_assert' needs C23",
    "body[0].body.body[0].body[0]: StaticAssertDeclaration without a message needs C23"]
# An `Error` stands where a statement or a member declaration may, and holds statements in its body, each checked and
# validated as one
ASSERT = lambda u: u.staticassertdeclaration(lambda s: s.keyword("static_assert").condition(a))  # noqa: E731
path = written(lambda u: u.error(lambda e: e.text("%").body([ASSERT, lambda u: u.whilestatement(lambda s: s.condition(
                   a))])),
               lambda u: u.simpledeclaration(lambda s: s.specifiers([lambda p: p.classspecifier(lambda k: k.key(
                   "struct").body(True).members(lambda m: m.memberlist(lambda m: m.members([lambda m: m.error(
                       lambda e: e.text("= ;"))]))))])))
assert Check.validate(STORE, path) == ["body[0]: invalid syntax '%'", "body[0].body[1]: a WhileStatement needs a body",
                                       "body[1].specifiers[0].members.members[0]: invalid syntax '= ;'"]
assert Check.check(STORE, path, "C17") == [
    "body[0].body[0]: StaticAssertDeclaration.keyword 'static_assert' needs C23",
    "body[0].body[0]: StaticAssertDeclaration without a message needs C23"]
with raises(AttributeError, match="memberdeclaration"):
    written(lambda u: u.error(lambda e: e.text("").body([lambda k: k.memberdeclaration(lambda d: d)])))
print("ok")

## CCHK-09 · Statements within terms: a statement expression's `block`, a struct's or a union's `members` and a conditional directive's `group`, checked and validated as statements, located through the term's place, to any depth; mbse-expressions' dialects do not read them

In [ ]:
path = parsed("int f(void) {\n"
              "    return ({ static_assert(1); 1; });\n"
              "}\n"
              "struct s {\n"
              "    struct { static_assert(1); int x; } inner;\n"
              "#ifdef A\n"
              "    static_assert(1);\n"
              "#endif\n"
              "};\n"
              "#if B\n"
              "#else\n"
              "int g = ({ int h(int a, int) { return a; } 0; });\n"
              "#endif\n")
assert Check.check(STORE, path, "C17") == [
    "body[0].body.body[0].value.block.body[0]: StaticAssertDeclaration.keyword 'static_assert' needs C23",
    "body[0].body.body[0].value.block.body[0]: StaticAssertDeclaration without a message needs C23",
    "body[1].specifiers[0].members.members[0].specifiers[0].members.members[0]: StaticAssertDeclaration.keyword "
    "'static_assert' needs C23",
    "body[1].specifiers[0].members.members[0].specifiers[0].members.members[0]: StaticAssertDeclaration without a "
    "message needs C23",
    "body[1].specifiers[0].members.members[1].directive.group.members[0]: StaticAssertDeclaration.keyword "
    "'static_assert' needs C23",
    "body[1].specifiers[0].members.members[1].directive.group.members[0]: StaticAssertDeclaration without a message "
    "needs C23",
    "body[2].directive.alternative.group.body[0].declarators[0].initializer.block.body[0]: FunctionDefinition with an "
    "unnamed parameter needs C23"], Check.check(STORE, path, "C17")
assert Check.check(STORE, path, "C23") == [] and Check.validate(STORE, path) == []
# Validated as statements: a kind's rules, an `Error` and a term's problems within a block, a member list or a group
path = written(
    lambda u: u.expressionstatement(lambda s: s.expression(lambda e: e.statementexpression(lambda x: x.block(
        lambda k: k.compoundstatement(lambda k: k.body([lambda u: u.whilestatement(lambda s: s.condition(a)),
                                                        lambda u: u.error(lambda e: e.text("x = = 1"))])))))),
    lambda u: u.simpledeclaration(lambda s: s.specifiers([lambda p: p.classspecifier(lambda k: k.key("union").body(
        True).members(lambda m: m.memberlist(lambda m: m.members([lambda m: m.memberdeclaration(lambda d: d.specifiers(
            [INT]).declarators([lambda d: d]))]))))])),
    lambda u: u.directive(lambda s: s.directive(lambda d: d.ifdefdirective(lambda i: i.name(lambda n: n.spelling(
        "A")).group(lambda g: g.group(lambda g: g.body([lambda u: u.expressionstatement(lambda s: s.expression(
            BAD))])))))))
assert Check.validate(STORE, path) == [
    "body[0].expression.block.body[0]: a WhileStatement needs a body",
    "body[0].expression.block.body[1]: invalid syntax 'x = = 1'",
    "body[1].specifiers[0].members.members[0].declarators[0]: a MemberDeclarator needs a declarator or a bitfield",
    "body[2].directive.group.body[0].expression: a memberexpression's operator must be '.' or '->', got '::'"]
# A term's dialect stops at its block (`Terms.boundaries`): what it holds is this module's to read
with STORE.transaction():
    unit = STORE.singleton(Model.WORKSPACE).files({"path": path})
    (_, statement), = Objects.properties(Objects.properties(unit)["body"].values[0]).items()
    expression = Objects.properties(statement)["expression"]
    assert Terms.kind_of(expression)[0] == "statementexpression" and CExpressions.DIALECT.validate(expression) == []
print("ok")

<a id="CCHK-10"></a>
## CCHK-10 · The ABI applied at the call: `validate` defers what depends on the target, `validate_for` checks it

As the user decided: "Of course C and C++ need a ABI, but if we can model programs in an abi-agnostic form and apply
the ABI at the last minute, that would be advantageous". A unit names no ABI, and a character constant holds its
escape's own number, unbounded (`L'\xffffffff'` is 4294967295). `validate` defers whether it fits its type;
`validate_for` applies an ABI: x86-64's `wchar_t` is `int`, which it does not fit; AArch64's is `unsigned int`, which
it does. An ABI not among `ABIS` is refused.

In [ ]:
from mbse.Programs import C
from support import raises

abi_store = C.store()
C.Standards.prepare(abi_store)
for abi in C.Check.ABIS:
    C.Check.prepare_abi(abi_store, abi)
abi_store.transaction(C.Parse.parse("wchar_t w = L'\\xffffffff';\nchar c = '\\xc8';\nint m = 'ab';\n", "abi.c"))
assert C.Check.validate(abi_store, "abi.c") == []  # deferred: no ABI applied
assert C.Check.validate_for(abi_store, "abi.c", "Expressions.SystemV.AMD64") == [
    "body[0].declarators[0].initializer: integer literal 4294967295 does not fit 'wchar_t'"]
assert C.Check.validate_for(abi_store, "abi.c", "Expressions.Arm.AAPCS64") == []
with abi_store.transaction():
    unit = abi_store.singleton(C.Parse.WORKSPACE).files({"path": "abi.c"})
    assert C.Check.validate_file_for(abi_store, unit, "Expressions.Arm.AAPCS64") == []
    with raises(LookupError, match="no ABI 'ILP32'"):
        C.Check.validate_file_for(abi_store, unit, "ILP32")
with raises(LookupError):
    C.Check.validate_for(abi_store, "missing.c", "Expressions.SystemV.AMD64")

## CCHK-11 · Grammar clauses, stage 3: each former `UNRECORDED` entry as data, an inferred type and an unnamed parameter the language's, a `u8` character C's; C23's removal of K&R definitions; an edition the data does not represent, C2y, and one before C11, guessed by designation for a grammar clause and a kind

As the user decided, on the shortcomings stage 2 found (mbse-basis, grammar clauses by version, "Decided in stage 3"):
"2. that's fine" (the grammar clause's further forms: `every` or `none` of a list's elements, `at_least` a count of
them, a grammar clause that `requires` others, and nested alternatives read as "any"), "3. yes" (removals as
`Forbids`) and "1. I already confirmed that" (an edition the data does not represent, guessed by its designation and
revision). So `Check` keeps no code for what a grammar clause cannot express: none is left.

In [ ]:
# Each former `UNRECORDED` entry, data: an inferred type, every specifier naming no type, requiring `auto`; an unnamed
# parameter of a definition's own, the declarator reaching the one nearest its name, whose parameters hold one without a
# declarator and either none of its specifiers `void` or two of them at least, alternatives; a `u8` character, C's
REACHING = ("functiondeclarator of a name with an unnamed parameter", *(
    f"{d}declarator reaching an unnamed parameter" for d in ("function", "pointer", "array", "parenthesized")))
with STORE.transaction():
    language = STORE.singleton(LANGUAGE)
    clauses = ExpressionStandards.grammar_clauses(language)
    said = lambda name: ExpressionStandards.grammar_clause(clauses[name])  # noqa: E731
    names = lambda found: tuple(map(ExpressionStandards.grammar_clause_name, found))  # noqa: E731
    inferred = said("SimpleDeclaration with an inferred type")
    assert inferred["quantifier"] == "every" and [kind.name.rsplit(".", 1)[-1] for kind in inferred["kinds"]] == [
        "DeclSpecifier", "CvQualifier", "StandardAttributeSpecifier", "AlignasSpecifier", "GnuAttributeSpecifier",
        "DeclspecSpecifier"]
    assert names(inferred["requires"]) == ("SimpleDeclaration.specifiers holding auto",)
    assert names(said("SimpleDeclaration.specifiers holding auto")["nested"]) == ("declspecifier.keyword 'auto'",)
    assert names(said("FunctionDefinition with an unnamed parameter")["nested"]) == REACHING
    assert all(names(said(name)["nested"]) == REACHING for name in REACHING[1:])  # through what holds a declarator
    assert names(said(REACHING[0])["requires"]) == ("functiondeclarator.parameters holding an unnamed parameter",)
    assert names(said("functiondeclarator.parameters holding an unnamed parameter")["nested"]) == (
        "parameterdeclaration unnamed, not void", "parameterdeclaration unnamed, of several specifiers")
    assert said("parameterdeclaration unnamed, not void")["present"] is False
    assert said("parameterdeclaration without void")["quantifier"] == "none"
    assert said("parameterdeclaration of several specifiers")["at_least"] == 2
    c = ExpressionStandards.grammar_clauses(STORE.singleton("Expressions.C.Expressions"))
    assert names(ExpressionStandards.grammar_clause(c["literal.prefix 'u8' of a character"])["requires"]) == (
        "literal of a character",)
# Through `check`, each located at the statement, as the code had it
needs("auto x = 1;\nstatic auto y = 2;\nconst auto z = 3;\n",
      "body[0]: SimpleDeclaration with an inferred type needs C23",
      "body[1]: SimpleDeclaration with an inferred type needs C23",
      "body[2]: SimpleDeclaration with an inferred type needs C23")
assert check("auto int x;\nauto struct s { int a; } t;\nstatic int y;\n", "C11") == []  # `auto`, a storage class
needs("int f(int, char *p) { return 0; }\nint g(const void) { return 0; }\nint (*h(int))(double) { return 0; }\n"
      "int (k)(int) { return 0; }\n",
      "body[0]: FunctionDefinition with an unnamed parameter needs C23",
      "body[1]: FunctionDefinition with an unnamed parameter needs C23",  # two specifiers, not a lone `void`
      "body[2]: FunctionDefinition with an unnamed parameter needs C23",  # its own, through a pointer, not its type's
      "body[3]: FunctionDefinition with an unnamed parameter needs C23")  # through parentheses
assert check("int f(void) { return 0; }\nint (*g(int a))(double) { return 0; }\nint h(int a, ...) { return a; }\n"
             "int k(int);\n", "C11") == []
needs("char c = u8'c';\nchar *s = u8\"s\";\n",
      "body[0].declarators[0].initializer: literal.prefix 'u8' of a character needs C23")  # a string is C11's
# What C23 removed: a K&R definition, its parameters declared in a declaration list, allowed by C11 and C17; its
# problem names the nearest edition that allows it
path = parsed("int f(a, b) int a; char b; { return a; }\nint g() { return 0; }\n")
assert Check.check(STORE, path, "C11") == Check.check(STORE, path, "C17") == []
assert Check.check(STORE, path, "C23") == ["body[0]: FunctionDefinition with a declaration list needs C17"]
# An edition the data does not represent, by its designation and its revision, without being stored: C2y guessed by
# C23's entries, flagged `after`, and C99 by C11's, flagged `before`; the language's and its dialects' alike
with STORE.transaction():
    language = STORE.singleton(LANGUAGE)
    clauses = ExpressionStandards.grammar_clauses(language)

    def guess(dialect, year, name):
        singleton = STORE.singleton(dialect)
        return ExpressionStandards.edition_allows(singleton, f"ISO/IEC 9899:{year}", ("year", year),
                                                  ExpressionStandards.grammar_clauses(singleton)[name])

    assert all(guess(LANGUAGE, 2029, name) and not guess(LANGUAGE, 1999, name) for name in NAMES)
    assert guess(LANGUAGE, 1999, REMOVED) and not guess(LANGUAGE, 2029, REMOVED)
    assert all(guess(LANGUAGE, 2029, name) and guess(LANGUAGE, 1999, name)
               for name in REACHING)  # nested: each allows them
    assert guess(LANGUAGE, 2018, REMOVED) and not guess(LANGUAGE, 2018, NAMES[0])  # covered: its own entries
    assert guess("Expressions.C.Expressions", 2029, "literal.prefix 'u8' of a character")
    assert not guess("Expressions.C.Types", 1999, "declspecifier.keyword 'thread_local'")
    kind = lambda year, name: ExpressionStandards.edition_has(  # noqa: E731
        language, f"ISO/IEC 9899:{year}", ("year", year), STORE.registered(f"{LANGUAGE}.{name}"))
    assert kind(2029, "AttributeDeclaration") and not kind(1999, "AttributeDeclaration") and kind(1999, "IfStatement")
    assert not kind(2029, "AsmDeclaration")  # GNU's, in none
    # another standard's designation is no neighbour: C++'s, of another number, guesses nothing, so allows each
    assert ExpressionStandards.edition_allows(language, "ISO/IEC 14882:2026", ("year", 2026), clauses[REMOVED])
# `check` takes the editions the language covers, as before: C2y is asked of the data, above
with raises(LookupError, match="covers no edition"):
    Check.check(STORE, path, "C2y")
print("ok")